**TransForm Customer Data**

**1.Remove Records With Null Data**

In [0]:
SELECT * FROM gizmobox_gr.bronze.v_customers
WHERE customer_id IS NOT NULL

**2.Remove Records Exact Duplicate Records**

In [0]:
SELECT DISTINCT * 
FROM gizmobox_gr.bronze.v_customers
ORDER BY customer_id

**3.Remove Duplicate Records Based On Created Timestamp**

In [0]:
CREATE OR REPLACE TEMPORARY VIEW v_customers_distinct AS
SELECT DISTINCT * 
FROM gizmobox_gr.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id

In [0]:
SELECT customer_id,MAX(created_timestamp)
FROM v_customers_distinct
GROUP BY customer_id

In [0]:
with cte_distinct as
(
SELECT customer_id,MAX(created_timestamp) as max_created_timestamp
FROM v_customers_distinct
GROUP BY customer_id
)
SELECT *
FROM v_customers_distinct c
join cte_distinct t on c.customer_id = t.customer_id and c.created_timestamp = t.max_created_timestamp

**4.CAST The Columns Into Correct Data Types**

In [0]:
with cte_distinct as
(
SELECT customer_id,MAX(created_timestamp) as max_created_timestamp
FROM v_customers_distinct
GROUP BY customer_id
)
SELECT CAST(t.max_created_timestamp AS TIMESTAMP),
c.customer_id,
c.customer_name,
CAST(c.date_of_birth AS TIMESTAMP) date_of_birth,
c.email,
c.telephone,
c.member_since,
c.customer_name,
c.FilePath
FROM v_customers_distinct c
join cte_distinct t on c.customer_id = t.customer_id and c.created_timestamp = t.max_created_timestamp

**5.Write Transformed Data To Silver Schema**

In [0]:
CREATE TABLE gizmobox_gr.silver.customers
AS
with cte_distinct as
(
SELECT customer_id,MAX(created_timestamp) as max_created_timestamp
FROM v_customers_distinct
GROUP BY customer_id
)
SELECT CAST(t.max_created_timestamp AS TIMESTAMP),
c.customer_id,
c.customer_name,
CAST(c.date_of_birth AS TIMESTAMP) date_of_birth,
c.email,
c.telephone,
c.member_since,
c.FilePath
FROM v_customers_distinct c
join cte_distinct t on c.customer_id = t.customer_id and c.created_timestamp = t.max_created_timestamp

In [0]:
SELECT * FROM gizmobox_gr.silver.customers